# Phase 02: Full Multimodal Run-Level Feature Extraction

This notebook extracts run-level aggregate features from all available modalities indexed by the Phase 01 audit. It treats `vrdataset/dataPackage` as read-only and writes all outputs only under `experiments/phase_02_full_multimodal_feature_extraction`.

The output level is one row per `subject_id`, `session_id`, `run_id`, `difficulty_level`, and `run_key`. This phase intentionally does not create the final cleaned modeling dataset, impute missing values, scale features, select features, or train models.

In [1]:

from pathlib import Path
import json
import pandas as pd
from IPython.display import display

ROOT = Path(r'E:\hdc-vr-pilot')
PHASE = ROOT / 'experiments' / 'phase_02_full_multimodal_feature_extraction'
P1 = ROOT / 'experiments' / 'phase_01_raw_data_modality_audit' / 'results'

inventory = pd.read_csv(P1 / 'raw_file_inventory.csv')
availability = pd.read_csv(P1 / 'run_modality_availability.csv')
modality_summary = pd.read_csv(P1 / 'modality_summary.csv')
unknown_review = pd.read_csv(P1 / 'unknown_files_for_review.csv')

print('Phase 01 input inventory summary')
print(f"Raw inventory files: {len(inventory):,}")
print(f"Run records: {availability['run_key'].nunique():,}")
print(f"Unknown files for review: {len(unknown_review):,}")
print('\nPhase 01 modality summary:')
display(modality_summary)
print('\nRun availability preview:')
display(availability.head())


Phase 01 input inventory summary
Raw inventory files: 9,003
Run records: 487
Unknown files for review: 908

Phase 01 modality summary:


,detected_modality,file_count
0,xplane,1461
1,eye_tracking,1443
2,respiration,1386
3,ecg,974
4,emg,974
5,eda,950
6,unknown,908
7,head_movement,487
8,performance,420



Run availability preview:


,subject_id,session_id,difficulty_level,run_id,run_key,has_eye_tracking,has_ecg,has_eda,has_emg,has_respiration,has_head_movement,has_xplane,has_control_input,has_performance,available_modality_count,available_modalities
0,sub-cp003,ses-20210206,level-000,run-001,sub-cp003_ses-20210206_level-000_run-001,True,True,True,True,True,True,True,False,False,7,ecg;eda;emg;eye_tracking;head_movement;respira...
1,sub-cp003,ses-20210206,level-000,run-002,sub-cp003_ses-20210206_level-000_run-002,True,True,True,True,True,True,True,False,False,7,ecg;eda;emg;eye_tracking;head_movement;respira...
2,sub-cp003,ses-20210206,level-01B,run-001,sub-cp003_ses-20210206_level-01B_run-001,True,True,True,True,True,True,True,False,True,8,ecg;eda;emg;eye_tracking;head_movement;perform...
3,sub-cp003,ses-20210206,level-01B,run-007,sub-cp003_ses-20210206_level-01B_run-007,True,True,True,True,True,True,True,False,True,8,ecg;eda;emg;eye_tracking;head_movement;perform...
4,sub-cp003,ses-20210206,level-01B,run-012,sub-cp003_ses-20210206_level-01B_run-012,True,True,True,True,True,True,True,False,True,8,ecg;eda;emg;eye_tracking;head_movement;perform...


## Execute Feature Extraction

The extraction code reads one stream file at a time, computes numeric aggregate features, logs failures without stopping the whole run, and keeps missing modality values as NaN. Performance metrics are retained but assigned to `performance_features` for leakage-sensitive handling in Phase 03.

In [2]:

import sys, importlib.util
sys.dont_write_bytecode = True
script_path = PHASE / 'scripts' / 'phase02_extract.py'
spec = importlib.util.spec_from_file_location('phase02_extract', script_path)
phase02 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(phase02)
report_path = PHASE / 'results' / 'extraction_report.json'
wide_path = PHASE / 'results' / 'full_multimodal_run_level_features.csv'
if report_path.exists() and wide_path.exists():
    report = json.loads(report_path.read_text(encoding='utf-8'))
    print('Existing completed extraction outputs found; loading the Phase 02 report instead of rereading raw streams.')
else:
    report = phase02.run_phase_02()
    print('Extraction complete.')
print(json.dumps(report['extraction_summary'], indent=2))


Existing completed extraction outputs found; loading the Phase 02 report instead of rereading raw streams.
{
  "processed_feature_files": 4945,
  "skipped_files": 4057,
  "failed_files": 0,
  "runs_in_output": 487,
  "features_in_wide_table_excluding_identifiers": 1247,
  "long_feature_rows": 599569,
  "modalities_extracted": [
    "eye_tracking",
    "ecg",
    "eda",
    "emg",
    "respiration",
    "head_movement",
    "xplane",
    "performance",
    "unknown"
  ]
}


## Output Review

These displays verify the required Phase 02 outputs: extracted modality list, processed run count, features per modality, preview rows, missing modality summary, failed file summary, feature group summary, limitations, and the Phase 03 handoff.

In [3]:

results = PHASE / 'results'
tables = PHASE / 'tables'
full = pd.read_csv(results / 'full_multimodal_run_level_features.csv')
long = pd.read_csv(results / 'feature_extraction_long_table.csv')
feature_groups = json.loads((results / 'feature_groups.json').read_text(encoding='utf-8'))
try:
    failed = pd.read_csv(results / 'failed_files.csv')
except pd.errors.EmptyDataError:
    failed = pd.DataFrame(columns=['file_name','run_key','detected_modality','error_type','error_message'])
fpm = pd.read_csv(tables / 'features_per_modality.csv')
runs_mod = pd.read_csv(tables / 'runs_with_extracted_modalities.csv')
missing = pd.read_csv(tables / 'missing_modalities_after_extraction.csv')
report = json.loads((results / 'extraction_report.json').read_text(encoding='utf-8'))

print('Extracted modality list:')
print(', '.join(report['extraction_summary']['modalities_extracted']))
print(f"\nNumber of runs successfully processed into output table: {len(full):,}")
print(f"Number of long-table feature rows: {len(long):,}")

print('\nFeatures per modality:')
display(fpm)

print('\nPreview of full_multimodal_run_level_features.csv:')
display(full.head())

print('\nRuns with extracted modalities preview:')
display(runs_mod.head())

print('\nMissing modality summary:')
display(missing['missing_modalities'].value_counts(dropna=False).reset_index(name='run_count').rename(columns={'index':'missing_modalities'}).head(20))

print('\nFailed file summary:')
if failed.empty:
    print('No files failed during extraction.')
else:
    display(failed[['file_name','run_key','detected_modality','error_type','error_message']].head(20))

print('\nFeature group summary:')
display(pd.DataFrame({'feature_group': list(feature_groups.keys()), 'feature_count': [len(v) for v in feature_groups.values()]}))

print('\nLimitations:')
for item in report['limitations']:
    print('-', item)

print('\nPhase 03 handoff:')
print('Final four-class labeling will be done in Phase 03. Phase 03 should construct modeling datasets with and without performance features, then handle imputation/scaling inside modeling pipelines.')


Extracted modality list:
eye_tracking, ecg, eda, emg, respiration, head_movement, xplane, performance, unknown

Number of runs successfully processed into output table: 487
Number of long-table feature rows: 599,569

Features per modality:


,modality,feature_group,feature_count,runs_with_features,non_missing_values
0,eye_tracking,eye_tracking_features,426,487,198622
1,ecg,physiological_features,57,487,27627
2,eda,physiological_features,44,475,20835
3,emg,physiological_features,84,487,40842
4,respiration,physiological_features,48,487,18825
5,head_movement,head_movement_features,159,487,76351
6,xplane,flight_parameter_features,328,487,155387
7,control_input,control_input_features,0,0,0
8,performance,performance_features,59,419,24721
9,unknown,unknown_features,42,454,19068



Preview of full_multimodal_run_level_features.csv:


,subject_id,session_id,run_id,difficulty_level,run_key,ecg_lslshimmerecg_duration,ecg_lslshimmerecg_ecg_projection_la_ra_mv_iqr,ecg_lslshimmerecg_ecg_projection_la_ra_mv_kurtosis,ecg_lslshimmerecg_ecg_projection_la_ra_mv_max,ecg_lslshimmerecg_ecg_projection_la_ra_mv_mean,...,xplane_lslxp11xpcac_aircraft_yaw_deg_missing_ratio,xplane_lslxp11xpcac_aircraft_yaw_deg_num_missing,xplane_lslxp11xpcac_aircraft_yaw_deg_range,xplane_lslxp11xpcac_aircraft_yaw_deg_rms,xplane_lslxp11xpcac_aircraft_yaw_deg_skew,xplane_lslxp11xpcac_aircraft_yaw_deg_slope,xplane_lslxp11xpcac_aircraft_yaw_deg_std,xplane_lslxp11xpcac_duration,xplane_lslxp11xpcac_sample_count,xplane_lslxp11xpcac_sampling_rate_estimate
0,sub-cp003,ses-20210206,run-001,level-000,sub-cp003_ses-20210206_level-000_run-001,324.413710,0.863729,-0.649131,-0.316374,-1.676774,...,0.0,0.0,0.000000,0.014609,NaN,0.000000,1.735157e-18,323.440259,2003.0,6.189706
1,sub-cp003,ses-20210206,run-002,level-000,sub-cp003_ses-20210206_level-000_run-002,328.206548,0.437634,0.001936,-3.152869,-4.275732,...,0.0,0.0,0.000000,0.000000,NaN,NaN,0.000000e+00,NaN,1.0,NaN
2,sub-cp003,ses-20210206,run-001,level-01B,sub-cp003_ses-20210206_level-01B_run-001,535.260675,0.535623,16.777445,5.340462,-1.481550,...,0.0,0.0,3.811121,0.437767,0.719148,-0.001203,4.336922e-01,534.491448,2407.0,4.501475
3,sub-cp003,ses-20210206,run-007,level-01B,sub-cp003_ses-20210206_level-01B_run-007,472.294350,0.281178,-0.415114,-3.523767,-4.357186,...,0.0,0.0,6.937821,0.556384,-0.077715,-0.005958,5.554413e-01,471.972053,2126.0,4.502385
4,sub-cp003,ses-20210206,run-012,level-01B,sub-cp003_ses-20210206_level-01B_run-012,472.847496,0.405901,-0.325430,-2.776779,-4.177590,...,0.0,0.0,3.484196,0.358065,0.559158,0.000458,3.580341e-01,471.853486,2125.0,4.501397



Runs with extracted modalities preview:


...


Missing modality summary:


,missing_modalities,run_count
0,control_input,408
1,control_input;performance,67
2,eda;control_input,11
3,eda;control_input;performance,1



Failed file summary:
No files failed during extraction.

Feature group summary:


,feature_group,feature_count
0,identifier_columns,5
1,physiological_features,233
2,eye_tracking_features,426
3,head_movement_features,159
4,flight_parameter_features,328
5,control_input_features,0
6,performance_features,59
7,unknown_features,42



Limitations:
- Run-level aggregate features only; no sliding-window modeling was performed.
- No final imputation, scaling, feature selection, four-class labeling, or model training was performed.
- ECG peak features are approximate; robust R-peak detection and HRV were not claimed.
- EDA and respiration peak counts are approximate, not validated event detection.
- Eye fixation/saccade sequence summaries were extracted where available, but detailed oculomotor event validation remains a limitation.
- Unknown torso accelerometer streams were kept in unknown_features rather than forced into known modalities.
- Performance features are retained and separately grouped for leakage-sensitive Phase 03 dataset versions.

Phase 03 handoff:
Final four-class labeling will be done in Phase 03. Phase 03 should construct modeling datasets with and without performance features, then handle imputation/scaling inside modeling pipelines.


## Saved Artifacts

All required outputs are saved under the Phase 02 experiment folder. The source `vrdataset` folder is not modified.

In [4]:

required = [
    results / 'full_multimodal_run_level_features.csv',
    results / 'feature_extraction_long_table.csv',
    results / 'feature_groups.json',
    results / 'failed_files.csv',
    results / 'extraction_report.json',
    tables / 'features_per_modality.csv',
    tables / 'runs_with_extracted_modalities.csv',
    tables / 'missing_modalities_after_extraction.csv',
    PHASE / 'figures' / 'extracted_features_per_modality.png',
    PHASE / 'figures' / 'runs_per_modality_after_extraction.png',
    PHASE / 'logs' / 'phase_02_log.txt',
    PHASE / 'README.md',
]
artifact_table = pd.DataFrame({'artifact': [str(p.relative_to(ROOT)) for p in required], 'exists': [p.exists() for p in required], 'size_bytes': [p.stat().st_size if p.exists() else 0 for p in required]})
display(artifact_table)


,artifact,exists,size_bytes
0,experiments\phase_02_full_multimodal_feature_e...,True,8848511
1,experiments\phase_02_full_multimodal_feature_e...,True,224856324
2,experiments\phase_02_full_multimodal_feature_e...,True,63934
3,experiments\phase_02_full_multimodal_feature_e...,True,96
4,experiments\phase_02_full_multimodal_feature_e...,True,5073
5,experiments\phase_02_full_multimodal_feature_e...,True,530
6,experiments\phase_02_full_multimodal_feature_e...,True,78354
7,experiments\phase_02_full_multimodal_feature_e...,True,49174
8,experiments\phase_02_full_multimodal_feature_e...,True,79646
9,experiments\phase_02_full_multimodal_feature_e...,True,78752
